In [1]:
from src.models.gpt import GPT, GPTConfig
from src.data.loader import build_loaders
import torch
from tqdm.auto import tqdm

In [2]:
train_loader, val_loader, test_loader = build_loaders()

In [3]:
losses = []
steps = []

# Overfit on a single batch
sanity_model = GPT(GPTConfig(vocab_size=10000))
sanity_optimizer = torch.optim.Adam(sanity_model.parameters(), lr=3e-3)
batch = next(iter(train_loader))
inputs, targets = batch['input_ids'], batch['labels']
for step in tqdm(range(1, 501), desc='Sanity Check'):
    outputs = sanity_model(inputs, targets)
    logits, loss = outputs['logits'], outputs['loss']
    sanity_optimizer.zero_grad()
    loss.backward()
    sanity_optimizer.step()
    losses.append(loss.item())
    steps.append(step)
    if step % 10 == 0:
        print(f'Step {step}, Loss: {loss.item()}')

Sanity Check:   0%|          | 0/500 [00:00<?, ?it/s]

Step 10, Loss: 5.7514472007751465
Step 20, Loss: 5.6441192626953125
Step 30, Loss: 5.626667022705078
Step 40, Loss: 5.576141357421875
Step 50, Loss: 5.439858436584473
Step 60, Loss: 5.219395637512207
Step 70, Loss: 4.9507527351379395
Step 80, Loss: 4.712418079376221
Step 90, Loss: 4.5050048828125
Step 100, Loss: 4.246324062347412
Step 110, Loss: 3.9735138416290283
Step 120, Loss: 3.6373116970062256
Step 130, Loss: 3.441276788711548
Step 140, Loss: 3.1080100536346436
Step 150, Loss: 2.726626396179199
Step 160, Loss: 2.39673113822937
Step 170, Loss: 2.0839028358459473
Step 180, Loss: 1.7659162282943726
Step 190, Loss: 1.4875702857971191
Step 200, Loss: 1.2227953672409058
Step 210, Loss: 1.0143840312957764
Step 220, Loss: 0.8300328850746155
Step 230, Loss: 0.6858532428741455
Step 240, Loss: 0.5449990034103394
Step 250, Loss: 0.45183151960372925
Step 260, Loss: 0.3599400818347931
Step 270, Loss: 0.29640713334083557
Step 280, Loss: 0.25507739186286926
Step 290, Loss: 0.21851493418216705
Ste

In [4]:
with torch.inference_mode():
    logits = sanity_model(input_ids=inputs, labels=targets)['logits']
    predictions = torch.argmax(logits, dim=-1)
    mask = targets.ne(-100)
    acc = (predictions[mask] == targets[mask]).float().mean().item()
    print("token_acc:", acc)

token_acc: 0.99072265625


In [ ]:
import matplotlib.pyplot as plt

plt.subplot(1, 2, 2)
plt.semilogy(steps, losses, 'r-', linewidth=1.5)
plt.xlabel('Training Step')
plt.ylabel('Loss (log scale)')
plt.title('Sanity Check: Loss Over Time (Log Scale)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(plots_dir / 'sanity_check_loss.png', dpi=150, bbox_inches='tight')
plt.show()